# Tuần 4: số liệu baseline phát hiện tấn công OOD

Quy trình: viết mã trên laptop, `git push`, notebook này chỉ `pull` rồi chạy. Kết quả ghi lên Google Drive, không vào repo.

Trước khi chạy: *Runtime → Change runtime type → T4 GPU*.

| Bước | Nội dung |
|---|---|
| 1 | Cài đặt, tải dữ liệu LYCOS-IDS2017 |
| 2 | Kiểm tra dữ liệu và chạy thử nhanh |
| 3 | Tái hiện số P0 của [16] bằng trọng số có sẵn (chỉ suy luận) |
| 4 | P0: 7 mô hình theo giao thức chia ngẫu nhiên |
| 5 | P1: loại từng họ tấn công khỏi tập huấn luyện |
| 6 | Tổng hợp bảng kết quả |

## 1. Cài đặt

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%env OUT_ROOT=/content/drive/MyDrive/project_3_runs
%env LYCOS_CSV=/content/lycos.csv
%env CACHE_DIR=/content
%env CLOSR_DIR=/content/project_3/third_party/CLOSR
import os
OUT = os.environ["OUT_ROOT"]
os.makedirs(OUT, exist_ok=True)

In [ ]:
# Chạy lại ô này sau mỗi lần `git push` từ laptop
%cd /content
!test -d project_3 && git -C project_3 pull || git clone https://github.com/hoangducanh1865/project_3
%cd /content/project_3
!pip install -q -r requirements.txt
!test -d third_party/CLOSR || git clone -q https://github.com/jackwilkie/CLOSR third_party/CLOSR

In [ ]:
# lycos.csv khoảng 860 MB (liên kết từ README của [16])
!test -f /content/lycos.csv || gdown 1PUMAbjz5L0MKiL3P-bEFTYYOcB6zPYB2 -O /content/lycos.csv
!ls -lh /content/lycos.csv

## 2. Kiểm tra dữ liệu và chạy thử

In [ ]:
import os, sys
os.chdir("/content/project_3"); sys.path.insert(0, "/content/project_3")
from hydra import initialize, compose
from src.data import load_csv, class_table

with initialize(version_base="1.3", config_path="conf"):
    cfg = compose(config_name="config", overrides=["protocol=p0"])
X, y = load_csv(cfg.dataset)
print(X.shape)
class_table(y)

Xem tên các lớp ở ô trên rồi chỉnh `HELDOUT` ở bước 5 cho khớp. Mỗi mục khớp không phân biệt hoa thường với một phần tên lớp, và phải khớp đúng một lớp.

In [ ]:
# Chạy thử vài giây: nếu ô này lỗi thì các bước sau cũng lỗi
!python -m src.run protocol=p0 model=mlp_msp eval_cap=5000 ++model.epochs=1

## 3. Tái hiện số P0 của [16]
Dùng trọng số có sẵn trong repo của [16], chỉ suy luận. Đối chiếu với README của repo:
- CLOSR: closed-set acc 0,995276; open-set AUC 0,974022; OpenAUC 0,969420
- CLAD: closed-set mean AUROC 0,999855; open-set mean AUROC 0,996627

In [ ]:
%cd /content/project_3/third_party/CLOSR
!mkdir -p data && cp /content/lycos.csv data/lycos.csv
!python eval_closr.py --device cuda 2>&1 | tee {OUT}/tai_hien_closr.txt
!python eval_clad.py --device cuda 2>&1 | tee {OUT}/tai_hien_clad.txt
%cd /content/project_3

## 4. P0: chia ngẫu nhiên
OOD là các lớp dưới 100 mẫu, giống [16]. CLAD/CLOSR ở đây huấn luyện ít vòng hơn bản gốc (xem `conf/model/*.yaml`), nên không phải số của bài.

In [ ]:
MODELS = "mlp_msp,mlp_knn,xgboost,iforest,autoencoder,clad,closr"
!python -m src.run -m protocol=p0 model={MODELS}

## 5. P1: loại từng họ tấn công khỏi tập huấn luyện
- Biến thể của chiến thuật đã biết: `slowloris` (còn các DoS khác), `ssh` (còn FTP-Patator).
- Chiến thuật mới: `bot` (Botnet), `portscan`.

In [ ]:
HELDOUT = "slowloris,ssh,bot,portscan"
!python -m src.run -m protocol=p1 protocol.heldout={HELDOUT} model={MODELS}

## 6. Tổng hợp

In [ ]:
!python -m src.report

In [ ]:
from IPython.display import Markdown
import os
Markdown(open(os.path.join(os.environ["OUT_ROOT"], "bao_cao_tuan4.md")).read())

Kết quả nằm ở `OUT_ROOT` trên Drive: `bao_cao_tuan4.md`, `results.csv`, `tai_hien_*.txt` và từng `result.json`. Gửi `bao_cao_tuan4.md` để viết báo cáo.